In [137]:
import pandas as pd
import numpy as np

In [138]:
df_lvl2 = pd.read_csv('df_level2.csv')
df_order = pd.read_csv('df_order.csv')
df_trade = pd.read_csv('df_trade.csv')

In [139]:
def convert_lvl2_time(df_lvl2: pd.DataFrame, date_col: str = "date", time_col: str = "time") -> pd.DataFrame:
    t = df_lvl2[time_col].astype(str).str.zfill(9)
    d = df_lvl2[date_col].astype(str)

    df_lvl2[time_col] = pd.to_datetime(
        d.str[:4] + '-' + d.str[4:6] + '-' + d.str[6:8] + ' ' +
        t.str[:2] + ':' + t.str[2:4] + ':' + t.str[4:6] + '.' + t.str[6:],
        format='%Y-%m-%d %H:%M:%S.%f'
    )

    return df_lvl2

In [140]:
df_lvl2 = df_lvl2.drop_duplicates(subset=['time'])
df_lvl2 = df_lvl2.ffill()
df_lvl2 = df_lvl2[df_lvl2["time"].astype(str).str.isdigit()].copy()
df_lvl2 = convert_lvl2_time(df_lvl2, date_col = 'date', time_col='time')
df_lvl2 = df_lvl2.sort_values(by='time').reset_index(drop=True)

df_order = df_order.drop_duplicates(subset=['id'])
df_order = df_order.dropna(subset=['order_time', 'price', 'quantity'])
df_order['order_time'] = pd.to_datetime(df_order['order_time'], format='mixed')
df_order['created_at'] = pd.to_datetime(df_order['created_at'], format='mixed')
df_order['order_time'] = df_order['order_time'].dt.tz_localize(None)
df_order['created_at'] = df_order['created_at'].dt.tz_localize(None)
df_order = df_order[(df_order['price'] > 0) & (df_order['quantity'] > 0)]
df_order = df_order.sort_values(by='order_time').reset_index(drop=True)

df_trade = df_trade.drop_duplicates(subset=['id'])
df_trade = df_trade.dropna(subset=['trade_time', 'price', 'quantity'])
df_trade['trade_time'] = pd.to_datetime(df_trade['trade_time'], format='mixed')
df_trade['created_at'] = pd.to_datetime(df_trade['created_at'], format='mixed')
df_trade['trade_time'] = df_trade['trade_time'].dt.tz_localize(None)
df_trade['created_at'] = df_trade['created_at'].dt.tz_localize(None)
df_trade = df_trade[(df_trade['price'] > 0) & (df_trade['quantity'] > 0)]
df_trade = df_trade.sort_values(by='trade_time').reset_index(drop=True)

In [141]:
def clean(order_book: dict, dec_places: int = 3) -> dict:
    return {
        "time": order_book["time"],
        "bid (price, quantity)": [(float(p), int(q)) for p, q in order_book["bid (price, quantity)"]],
        "ask (price, quantity)": [(float(p), int(q)) for p, q in order_book["ask (price, quantity)"]],
        "total_bid_volume": int(order_book["total_bid_volume"]),
        "total_ask_volume": int(order_book["total_ask_volume"]),
        "last_price": round(float(order_book["last_price"]), dec_places),
        "total_volume": int(order_book["total_volume"]),
        "total_amount": round(float(order_book["total_amount"]), dec_places),
    }

In [142]:
def get_orderbook(time: str, brackets: int = 10) -> dict:
    dt = pd.to_datetime(time, format='mixed')

    # 起始lvl2快照
    base = df_lvl2[df_lvl2['time'] <= dt].iloc[-1]
    
    last_price = base["last"]
    total_volume = base["volume"]
    total_amount = base["amount"]

    order_book = {"bid": {}, "ask": {}}
    for i in range(1, brackets + 1):
        bp, bv = base[f"bp{i}"], base[f"bv{i}"]
        ap, av = base[f"ap{i}"], base[f"av{i}"]
        order_book['bid'][bp] = bv
        order_book['ask'][ap] = av
    
    # 选择相关订单和交易
    t0 = base['time']
    orders = df_order[(df_order['order_time'] >= t0 ) & (df_order['order_time'] <= dt)]
    trades = df_trade[(df_trade['trade_time'] >= t0 ) & (df_trade['trade_time'] <= dt)]

    # 按时间合并订单和交易事件
    orders = orders.assign(type='order', ts=orders['order_time'])
    trades = trades.assign(type='trade', ts=trades['trade_time'])

    # 逐单处理事件，更新订单簿和统计数据
    events = pd.concat([orders, trades]).sort_values('ts')
    for _, event in events.iterrows():
        if event['type'] == 'order':
            book = order_book['bid'] if event['side'] == 'buy' else order_book['ask']  # 买单更新买盘，卖单更新卖盘

            # 新增加或取消减
            if event['order_type'] == 'place':
                book[event['price']] = book.get(event['price'], 0) + event['quantity']
            elif event['price'] in book: # cancel
                book[event['price']] -= event['quantity']
                if book[event['price']] <= 0:
                    del book[event['price']]
        else: # trade
            price, quantity = event['price'], event['quantity']
            last_price = price
            total_volume += quantity
            total_amount += price * quantity
            
            if event['other_info'] == 'external(buy)':
                # 主动买 → 吃卖盘（ask）
                if price in order_book['ask']:
                    order_book['ask'][price] -= quantity
                    if order_book['ask'][price] <= 0:
                        del order_book['ask'][price]

            else: # internal(sell)
                # 主动卖 → 吃买盘（bid）
                if price in order_book['bid']:
                    order_book['bid'][price] -= quantity
                    if order_book['bid'][price] <= 0:
                        del order_book['bid'][price]

    result = {
        "time": time,
        "bid (price, quantity)": sorted(order_book['bid'].items(), reverse=True)[:10],
        "ask (price, quantity)": sorted(order_book['ask'].items())[:10],
        "total_bid_volume": sum(order_book['bid'].values()),
        "total_ask_volume": sum(order_book['ask'].values()),
        "last_price": last_price,
        "total_volume": total_volume,
        "total_amount": total_amount
    }

    return clean(result)

In [145]:
df_lvl2

,time,status,pre_close,open,high,low,last,ap1,ap2,ap3,...,high_limited,low_limited,prefix,syl1,syl2,sd2,trading_phase_code,pre_iopv,symbol,date
0,2024-02-07 08:45:01,0,2.351,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,2.586,2.116,b'',0,0,0,b'S 10',0.0,SH.510050,20240207
1,2024-02-07 08:45:04,0,2.351,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,2.586,2.116,b'',0,0,0,b'S 10',0.0,SH.510050,20240207
2,2024-02-07 08:45:07,0,2.351,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,2.586,2.116,b'',0,0,0,b'S 10',0.0,SH.510050,20240207
3,2024-02-07 08:45:10,0,2.351,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,2.586,2.116,b'',0,0,0,b'S 10',0.0,SH.510050,20240207
4,2024-02-07 08:45:13,0,2.351,0.000,0.000,0.000,0.00,0.000,0.000,0.000,...,2.586,2.116,b'',0,0,0,b'S 10',0.0,SH.510050,20240207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8161,2024-02-07 15:57:52,0,2.351,2.355,2.381,2.338,2.38,2.381,2.382,2.383,...,2.586,2.116,b'',0,0,0,b'E110',0.0,SH.510050,20240207
8162,2024-02-07 15:58:22,0,2.351,2.355,2.381,2.338,2.38,2.381,2.382,2.383,...,2.586,2.116,b'',0,0,0,b'E110',0.0,SH.510050,20240207
8163,2024-02-07 15:58:52,0,2.351,2.355,2.381,2.338,2.38,2.381,2.382,2.383,...,2.586,2.116,b'',0,0,0,b'E110',0.0,SH.510050,20240207
8164,2024-02-07 15:59:22,0,2.351,2.355,2.381,2.338,2.38,2.381,2.382,2.383,...,2.586,2.116,b'',0,0,0,b'E110',0.0,SH.510050,20240207


In [164]:
ob = get_orderbook('2024-02-07 09:37:19')

In [165]:
ob

{'time': '2024-02-07 09:37:19',
 'bid (price, quantity)': [(2.34, 771900),
  (2.339, 3294500),
  (2.338, 2441400),
  (2.337, 2725700),
  (2.336, 1712100),
  (2.335, 1795000),
  (2.334, 323300),
  (2.333, 1478600),
  (2.332, 772500),
  (2.331, 1840800)],
 'ask (price, quantity)': [(2.341, 133100),
  (2.342, 1009700),
  (2.343, 2217200),
  (2.344, 2467200),
  (2.345, 2191069),
  (2.346, 351200),
  (2.347, 340700),
  (2.348, 731600),
  (2.349, 418800),
  (2.35, 1080700)],
 'total_bid_volume': 17155800,
 'total_ask_volume': 10941269,
 'last_price': 2.341,
 'total_volume': 140202100,
 'total_amount': 32898.237}